# WSN Palisades — terrain/vegetation/solar-aware sensor placement

This notebook walks through the full pipeline for selecting K wireless-sensor
locations under a realistic visibility model (terrain occlusion + canopy
attenuation) and a clearsky solar model, comparing four selection methods:

1. **Random** — feasible-only baseline.
2. **Greedy** — single-pass, weighted-sum greedy.
3. **Simple NSGA-III** — pymoo with random feasible sampling and a
   `GeometricRepair` operator.
4. **Seeded NSGA-III** — same, but the initial population is seeded with
   greedy solutions across a balanced weight simplex.

Pipeline: **AOI → DEM/DSM/CHM → directional γ(θ) and effective radius →
coverage masks → optimization → IEEE figures**.

The expensive precompute and full K-sweep are not re-run here — they take
hours. Instead we load the canonical saved result and rebuild figures.
For full reproduction see `scripts/run_ksweep.py`.


In [ ]:
# Imports — note the package is editable-installed (`pip install -e .`)
from pathlib import Path

from wsn_palisades import SensorParams, SolarParams
from wsn_palisades import candidates as wp_candidates
from wsn_palisades import persistence as wp_persistence
from wsn_palisades import plotting as wp_plotting
from wsn_palisades import maps as wp_maps

REPO = Path("..").resolve()
RESULTS = REPO / "results"
SAMPLE = REPO / "sample"


## 1. AOI — load the canonical Palisades polygon

In [ ]:
aoi_poly = wp_candidates.load_aoi(SAMPLE / "aoi_palisades.geojson")
print("AOI bounds:", aoi_poly.bounds)
print(f"AOI area:   {aoi_poly.area:.6f} (deg^2)")


## 2. (Optional) Build the DEM/DSM/CHM manager and precompute scenario packs

The cells below are *commented out* in the demo path — they require the
1–2 GB GeoTIFFs in `data/` (fetch them with `python scripts/fetch_data.py`)
and the precompute itself takes ~10 minutes per scenario.

For the demo we skip straight to the saved K-sweep results.


In [ ]:
# from wsn_palisades.surfaces import DEMManager, warp_surfaces_to_utm
# SP = SensorParams()
# solar = SolarParams()
#
# dem = DEMManager.from_files(
#     aoi_poly,
#     dtm_path=str(REPO / "data" / "palisadesoutput.dtm.tif"),
#     dsm_path=str(REPO / "data" / "palisadesoutput.dsm.tif"),
#     chm_path=str(REPO / "data" / "palisadesCHM.tif"),
# )
# warp_surfaces_to_utm(dem, aoi_poly, target_res_m=2.0)
#
# packs_flat, packs_dem, packs_dsmchm = wp_candidates.precompute_all(
#     aoi_poly, dem, SP, grid_size=30, cov_grid_size=80, solar_params=solar
# )


## 3. Load the canonical saved K-sweep

In [ ]:
res_k_sweep = wp_persistence.load_ksweep(RESULTS / "res_k_sweep_k=10to60_PalisadesFinal.pkl.gz")
print("Scenarios:", sorted(res_k_sweep.keys()))
for s in sorted(res_k_sweep.keys()):
    print(f"  {s}: K = {sorted(res_k_sweep[s].keys())}")


In [ ]:
import pandas as pd
df_k_sweep = pd.read_csv(RESULTS / "csv" / "df_k_sweep_k=10to60_PalisadesFinal.csv")
df_k_sweep.head()


## 4. IEEE Figure 1 — Coverage vs K

In [ ]:
fig = wp_plotting.plot_coverage_vs_k(df_k_sweep)
fig


## 5. IEEE Figure 2 — Distance to ideal vs K

In [ ]:
fig = wp_plotting.plot_distance_to_ideal_vs_k(df_k_sweep)
fig


## 6. IEEE Figure 3 — Pareto trade-off (coverage vs visibility)

In [ ]:
fig = wp_plotting.plot_pareto_tradeoff(df_k_sweep, x="coverage_pct", y="gamma_mean")
fig


## 7. Summary panel — all four metrics, all scenarios

In [ ]:
fig = wp_plotting.plot_summary_four_panel(df_k_sweep)
fig


## 8. Interactive map of a chosen sensor placement

In [ ]:
map_widget = wp_maps.build_ipyleaflet_map(aoi_poly, res_k_sweep, grid_size=30)
map_widget


## Optimizer comparison — what we observe across terrains and K

Across all terrain types (FLAT, DEM, DSM/CHM) and sensor counts, the four
optimizers show distinct patterns:

1. **Random** is consistently the weakest method — lowest coverage, γ̄,
   spacing, and solar. Serves as a baseline only.

2. **Greedy** is a very strong single-pass heuristic, especially on
   DEM and DSM/CHM. Typically gets high coverage, high solar, and
   strong γ̄ — but often collapses mean spacing (sensors cluster).

3. **Simple NSGA-III** consistently beats random and often gets the
   best coverage on FLAT (smooth landscape). On DEM and DSM/CHM it
   lags greedy and seeded NSGA on solar and γ̄ because it has no
   problem-specific seeding.

4. **Seeded NSGA-III** is the strongest multi-objective optimizer
   overall. On FLAT it finds high-solar solutions with competitive
   coverage; on terrain it matches or exceeds greedy across multiple
   objectives.

See the IEEE-style figures in `results/figures/` for the publication
versions of these plots.
